# Client Satisfaction and Loyalty Analysis

## Background

This analysis examines customer satisfaction across different client segments and identifies which service dimensions are most closely associated with customer loyalty.

### What is NPS?

**NPS** stands for **Net Promoter Score**. It measures how willing a customer is to recommend a company to others.

Customers normally provide a rating from 0 to 10:

* **Promoters:** Ratings of 9–10
* **Passives:** Ratings of 7–8
* **Detractors:** Ratings of 0–6

The formal Net Promoter Score is calculated as:

**NPS = Percentage of Promoters − Percentage of Detractors**

However, this dataset contains each client's individual **NPS Rating from 0 to 10**. Therefore, the visualisations use the **average NPS Rating**, rather than the formal NPS percentage. A higher average rating suggests stronger customer advocacy and loyalty.

### What is NPO?

**NPO** stands for **Non-Profit Organisation**. These organisations operate mainly to support a social cause or public purpose rather than to generate profits for owners or shareholders.

The three client types analysed are:

* **Govt:** Government organisations
* **NPO:** Non-profit organisations
* **Private:** Privately owned businesses

## Variables Used

| Variable                     | Meaning                                                                                                                                    |
| ---------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------ |
| **CLIENT ID**                | A unique identifier assigned to each client.                                                                                               |
| **YEAR**                     | The year in which the financial and survey information was recorded, from 2021 to 2025.                                                    |
| **TYPE**                     | The client's organisation type: Govt, NPO or Private.                                                                                      |
| **PRESALES AND PARTNERSHIP** | A 1–5 rating of the company's engagement before a project begins and its ability to maintain a strong working relationship.                |
| **TECHNICAL EXPERTISE**      | A 1–5 rating of the company's technical knowledge, skills and ability to provide effective solutions.                                      |
| **PROJECT DELIVERY**         | A 1–5 rating of how effectively the company plans, manages and completes projects.                                                         |
| **POST-SALES SUPPORT**       | A 1–5 rating of the assistance provided after project completion, including issue resolution and ongoing support.                          |
| **NPS RATING**               | A 0–10 rating representing the client's willingness to recommend the company. It is used as an indicator of customer advocacy and loyalty. |

## Derived Measures

The following measures are calculated from the original variables:

* **Average Service Rating:** The mean rating for each service dimension within a client type and period.
* **Average NPS Rating:** The mean NPS Rating within a client type and period.
* **Record Count:** The number of client-year observations included in each average.
* **Spearman Correlation:** Measures how strongly each service dimension is associated with NPS Rating. A higher positive value means that higher service ratings generally occur alongside higher NPS ratings.

## Unit of Analysis

Each row represents one **client-year observation**. A client may appear in multiple years, but there should only be one record for the same Client ID and Year combination.

## Analysis Objective

The objective of this notebook is to:

1. Compare service satisfaction across Govt, NPO and Private clients.
2. examine how satisfaction and NPS ratings change from 2021 to 2025;
3. identify client segments with consistently high or low satisfaction; and
4. determine which service dimensions have the strongest association with customer loyalty.

## Link to the Main Problem Statement

The team's main problem is to determine which client segments generate the highest gross profit while maintaining strong customer satisfaction.

This notebook provides the **customer satisfaction component** by identifying:

* which client segments report the strongest satisfaction and loyalty;
* which segments may require additional attention; and
* which service dimensions should be prioritised for improvement.

These findings can then be combined with the profitability analysis to identify client segments that provide both **high financial value** and **strong customer outcomes**.


## Data preparation and validation

The analysis uses the cleaned, merged client dataset. Each row represents a **client-year observation**, so the same client may legitimately appear in different years.

- Service dimensions are rated on a **1–5 scale**.
- NPS Rating is recorded on a **0–10 scale**.
- Client Type is the main segmentation variable used in these visualisations.

The checks below confirm the dataset shape, inspect the available fields and verify that there is no more than one record for the same Client ID and Year combination.

In [1]:
import pandas as pd
import plotly.express as px

df = pd.read_excel("merged_cleaned.xlsx")

print("Shape:", df.shape)
print(
    "Duplicate client-year records:",
    df.duplicated(subset=["CLIENT ID", "YEAR"]).sum()
)

display(df.head())

Shape: (392, 16)
Duplicate client-year records: 0


,CLIENT ID,TYPE,COMMENCEMENT DATE,STAFF STRENGTH,SECTOR,COUNTRY,YEAR,PRESALES AND PARTNERSHIP,TECHNICAL EXPERTISE,PROJECT DELIVERY,POST-SALES SUPPORT,NPS RATING,REVENUE,HARDWARE,SOFTWARE,MANPOWER
0,G1-001,Govt,2020-01-19 00:00:00,> 200,Finance,Singapore,2021,3,4,4,5,8,148180,17321,25562,41201
1,G1-001,Govt,2020-01-19 00:00:00,> 200,Finance,Singapore,2022,5,5,3,4,9,197313,23604,32875,50085
2,G1-001,Govt,2020-01-19 00:00:00,> 200,Finance,Singapore,2023,5,4,3,4,7,349899,51332,71695,89089
3,G1-001,Govt,2020-01-19 00:00:00,> 200,Finance,Singapore,2024,5,5,3,4,8,268821,40969,56452,76545
4,G1-001,Govt,2020-01-19 00:00:00,> 200,Finance,Singapore,2025,5,4,5,5,10,138115,14516,20889,28968


In [2]:
print(
    "Duplicate client-year records:",
    df.duplicated(subset=["CLIENT ID", "YEAR"]).sum()
)

Duplicate client-year records: 0


### Validation result

The cleaned dataset contains **392 client-year observations and 16 variables**, with **0 duplicate Client ID–Year records**. This supports fair yearly comparisons without double-counting the same client within a year.

The next two cells define the four service dimensions and calculate an initial client-type summary before the interactive visualisations are created.


In [3]:
service_columns = [
    "PRESALES AND PARTNERSHIP",
    "TECHNICAL EXPERTISE",
    "PROJECT DELIVERY",
    "POST-SALES SUPPORT"
]

In [4]:
type_summary = (
    df.groupby("TYPE")[service_columns]
      .mean()
      .round(2)
      .reset_index()
)

display(type_summary)

,TYPE,PRESALES AND PARTNERSHIP,TECHNICAL EXPERTISE,PROJECT DELIVERY,POST-SALES SUPPORT
0,Govt,4.09,3.94,4.35,3.91
1,NPO,3.76,3.88,4.26,4.12
2,Private,3.96,3.92,4.01,3.80


## Visualisation 1 — Service Ratings by Client Type and Period

**Purpose:** Compare the four service dimensions across Govt, NPO and Private clients, and observe whether their ratings change from 2021 to 2025.

**How to interact:**

- Press **Play** to animate the yearly changes.
- Drag the **period slider** to select a specific year or the Overall view.
- Hover over a bar to view the average rating and number of records.
- Click a legend item to show or hide a service dimension.

The y-axis remains fixed from 0 to 5 so that changes between periods are visually comparable.


In [5]:
import pandas as pd
import plotly.express as px


# -------------------------------------------------
# 1. Settings
# -------------------------------------------------

service_columns = [
    "PRESALES AND PARTNERSHIP",
    "TECHNICAL EXPERTISE",
    "PROJECT DELIVERY",
    "POST-SALES SUPPORT"
]

service_label_map = {
    "PRESALES AND PARTNERSHIP": "Presales & Partnership",
    "TECHNICAL EXPERTISE": "Technical Expertise",
    "PROJECT DELIVERY": "Project Delivery",
    "POST-SALES SUPPORT": "Post-Sales Support"
}

service_order = list(service_label_map.values())
type_order = ["Govt", "NPO", "Private"]

service_colors = {
    "Presales & Partnership": "#636EFA",
    "Technical Expertise": "#EF553B",
    "Project Delivery": "#00CC96",
    "Post-Sales Support": "#AB63FA"
}


# -------------------------------------------------
# 2. Calculate overall averages
# -------------------------------------------------

overall_summary = (
    df.groupby("TYPE", as_index=False)[service_columns]
      .mean()
      .round(2)
)

overall_counts = (
    df.groupby("TYPE")
      .size()
      .rename("Records")
      .reset_index()
)

overall_summary = overall_summary.merge(
    overall_counts,
    on="TYPE"
)

overall_long = overall_summary.melt(
    id_vars=["TYPE", "Records"],
    value_vars=service_columns,
    var_name="Service Dimension",
    value_name="Average Rating"
)

overall_long["Service Dimension"] = (
    overall_long["Service Dimension"]
    .replace(service_label_map)
)

overall_long["Period"] = "Overall"


# -------------------------------------------------
# 3. Calculate yearly averages
# -------------------------------------------------

yearly_summary = (
    df.groupby(["YEAR", "TYPE"], as_index=False)[service_columns]
      .mean()
      .round(2)
)

yearly_counts = (
    df.groupby(["YEAR", "TYPE"])
      .size()
      .rename("Records")
      .reset_index()
)

yearly_summary = yearly_summary.merge(
    yearly_counts,
    on=["YEAR", "TYPE"]
)

yearly_long = yearly_summary.melt(
    id_vars=["YEAR", "TYPE", "Records"],
    value_vars=service_columns,
    var_name="Service Dimension",
    value_name="Average Rating"
)

yearly_long["Service Dimension"] = (
    yearly_long["Service Dimension"]
    .replace(service_label_map)
)

yearly_long["Period"] = (
    yearly_long["YEAR"]
    .astype(int)
    .astype(str)
)


# -------------------------------------------------
# 4. Combine yearly and overall results
# -------------------------------------------------

animation_data = pd.concat(
    [
        yearly_long.drop(columns="YEAR"),
        overall_long
    ],
    ignore_index=True
)

animation_data["Animation Group"] = (
    animation_data["TYPE"]
    + " — "
    + animation_data["Service Dimension"]
)

period_order = [
    str(int(year))
    for year in sorted(df["YEAR"].unique())
] + ["Overall"]


# -------------------------------------------------
# 5. Create animated chart
# -------------------------------------------------

fig = px.bar(
    animation_data,
    x="TYPE",
    y="Average Rating",
    color="Service Dimension",
    barmode="group",

    animation_frame="Period",
    animation_group="Animation Group",

    text="Average Rating",
    custom_data=["Records"],

    category_orders={
        "Period": period_order,
        "TYPE": type_order,
        "Service Dimension": service_order
    },

    color_discrete_map=service_colors,

    labels={
        "TYPE": "Client Type",
        "Average Rating": "Average Rating"
    },

    range_y=[0, 5.4],

    title="Service Ratings by Client Type: 2021–2025 and Overall"
)


# -------------------------------------------------
# 6. Hover and data-label formatting
# -------------------------------------------------

trace_format = dict(
    texttemplate="%{y:.2f}",
    textposition="outside",
    cliponaxis=False,

    hovertemplate=(
        "<b>Client Type: %{x}</b><br>"
        "Service: %{fullData.name}<br>"
        "Average Rating: %{y:.2f} / 5<br>"
        "Records Used: %{customdata[0]}"
        "<extra></extra>"
    )
)

fig.update_traces(**trace_format)

# Apply the same formatting to every animation frame
for frame in fig.frames:
    for trace in frame.data:
        trace.update(**trace_format)


# -------------------------------------------------
# 7. Adjust animation speed
# -------------------------------------------------

play_settings = (
    fig.layout.updatemenus[0]
       .buttons[0]
       .args[1]
)

play_settings["frame"] = {
    "duration": 1400,
    "redraw": True
}

play_settings["transition"] = {
    "duration": 700,
    "easing": "cubic-in-out"
}

play_settings["fromcurrent"] = True

fig.layout.sliders[0].currentvalue.prefix = "Period: "
fig.layout.sliders[0].pad.t = 50


# -------------------------------------------------
# 8. Final formatting
# -------------------------------------------------

fig.update_layout(
    template="plotly_white",

    title=dict(
        x=0.5,
        xanchor="center"
    ),

    xaxis_title="Client Type",
    yaxis_title="Average Rating",

    legend_title="Service Dimension",

    transition=dict(
        duration=700,
        easing="cubic-in-out"
    ),

    margin=dict(
        t=100,
        b=130
    )
)

fig.update_yaxes(
    range=[0, 5.4],
    dtick=1
)


# -------------------------------------------------
# 9. Display graph
# -------------------------------------------------

fig.show(
    config={
        "displayModeBar": True,
        "displaylogo": False,
        "scrollZoom": True,
        "responsive": True,

        "toImageButtonOptions": {
            "format": "png",
            "filename": "service_ratings_story",
            "scale": 2
        }
    }
)

### Key findings from Visualisation 1

- **Project Delivery is the strongest overall service dimension** for every client type: Govt 4.35, NPO 4.26 and Private 4.01.
- Project Delivery reaches its highest displayed values in **2025**: Govt 4.75, NPO 4.54 and Private 4.18.
- **NPO Presales & Partnership** is comparatively weak overall at 3.76, suggesting a possible early-stage relationship gap.
- **Private Post-Sales Support** is the lowest overall result for Private clients at 3.80, making it a practical improvement area.

**Business implication:** The company should protect its strong Project Delivery performance while tailoring improvements by segment—particularly Presales & Partnership for NPO clients and Post-Sales Support for Private clients.

**Caution:** These are averages. Use the record counts in the hover information when comparing segments because the groups are not equally sized.

## Visualisation 2 — Average NPS Rating by Client Type and Year

**Purpose:** Examine customer advocacy and loyalty patterns across client types and periods. The heatmap makes high and low combinations easy to locate, while the printed values support precise comparison.

**How to interact:** Hover over a cell to view its client type, period, average NPS rating and record count.

> **Measurement note:** This chart shows the average **NPS Rating** on a 0–10 survey scale. It is not the formal Net Promoter Score, which is calculated as the percentage of Promoters minus the percentage of Detractors.

In [6]:
import pandas as pd
import plotly.graph_objects as go


# -------------------------------------------------
# 1. Set display order
# -------------------------------------------------

type_order = [
    client_type
    for client_type in ["Govt", "NPO", "Private"]
    if client_type in df["TYPE"].unique()
]

years = sorted(
    df["YEAR"].dropna().astype(int).unique()
)


# -------------------------------------------------
# 2. Calculate average NPS rating
# -------------------------------------------------

nps_heatmap = (
    df.groupby(["TYPE", "YEAR"])["NPS RATING"]
      .mean()
      .round(2)
      .unstack("YEAR")
      .reindex(type_order)
)

nps_heatmap.columns = [
    str(int(year))
    for year in nps_heatmap.columns
]

nps_heatmap["Overall"] = (
    df.groupby("TYPE")["NPS RATING"]
      .mean()
      .round(2)
      .reindex(type_order)
)


# -------------------------------------------------
# 3. Calculate number of records for hover
# -------------------------------------------------

record_heatmap = (
    df.groupby(["TYPE", "YEAR"])
      .size()
      .unstack("YEAR")
      .reindex(type_order)
)

record_heatmap.columns = [
    str(int(year))
    for year in record_heatmap.columns
]

record_heatmap["Overall"] = (
    df.groupby("TYPE")
      .size()
      .reindex(type_order)
)


# -------------------------------------------------
# 4. Create Graph Objects heatmap
# -------------------------------------------------

fig2 = go.Figure(
    data=go.Heatmap(
        x=nps_heatmap.columns,
        y=nps_heatmap.index,
        z=nps_heatmap.values,

        text=nps_heatmap.values,
        texttemplate="%{text:.2f}",
        textfont=dict(
            size=16,
            color="#17324D"
        ),

        customdata=record_heatmap.values,

        zmin=7,
        zmax=9,

        colorscale=[
            [0.00, "#E76F51"],
            [0.25, "#F4A261"],
            [0.50, "#F6D365"],
            [0.75, "#82C9A5"],
            [1.00, "#2A9D8F"]
        ],

        colorbar=dict(
            title="Average NPS<br>Rating",
            tickvals=[7, 7.5, 8, 8.5, 9],
            ticktext=["7.0", "7.5", "8.0", "8.5", "9.0+"],
            thickness=18,
            len=0.85
        ),

        xgap=5,
        ygap=5,

        hovertemplate=(
            "<b>Client Type: %{y}</b><br>"
            "Period: %{x}<br>"
            "Average NPS Rating: %{z:.2f} / 10<br>"
            "Records Used: %{customdata}"
            "<extra></extra>"
        )
    )
)

# -------------------------------------------------
# 5. Format graph
# -------------------------------------------------

fig2.update_layout(
    title=dict(
        text="Average NPS Rating by Client Type and Year",
        x=0.5,
        xanchor="center"
    ),

    xaxis=dict(
        title="Period",
        side="bottom",
        type="category"
    ),

    yaxis=dict(
        title="Client Type",
        autorange="reversed",
        type="category"
    ),

    template="plotly_white",

    hoverlabel=dict(
        bgcolor="white",
        font_size=13
    ),

    margin=dict(
        l=100,
        r=120,
        t=90,
        b=80
    ),

    height=430
)


# -------------------------------------------------
# 6. Display graph
# -------------------------------------------------

fig2.show(
    config={
        "displayModeBar": True,
        "displaylogo": False,
        "responsive": True,

        "toImageButtonOptions": {
            "format": "png",
            "filename": "nps_rating_heatmap",
            "scale": 2
        }
    }
)

### Key findings from Visualisation 2

- NPO has the highest overall average NPS Rating at **8.12**, closely followed by Govt at **8.11**; Private is lower at **7.81**.
- **2025 is the strongest NPS period for all three client types**: Govt 8.65, NPO 8.54 and Private 8.02.
- Govt records the highest individual client-type/year value at **8.65 in 2025**, indicating especially strong recent advocacy among Government clients.

**Business implication:** Private clients show the clearest overall loyalty-improvement opportunity. The strong 2025 results across all segments should also be investigated to identify practices worth maintaining.

**Caution:** The colour scale is focused on the 7–9 range to make differences visible, so the colour contrast can make relatively small numerical gaps look larger. Read the displayed values alongside the colours.


## Visualisation 3 — Service Dimensions Associated with NPS Rating

**Purpose:** Identify which service dimensions are most strongly associated with customer loyalty overall and within each client type.

The chart uses **Spearman correlation**, which is suitable for examining whether higher ordinal service ratings generally move together with higher NPS ratings without assuming a perfectly linear relationship.

**How to interact:** Use the dropdown menu to switch between Overall, Govt, NPO and Private results. Hover over a bar to view the exact correlation and its interpretation band.


In [7]:
import pandas as pd
import plotly.graph_objects as go


# -------------------------------------------------
# 1. Variables and labels
# -------------------------------------------------

service_columns = [
    "PRESALES AND PARTNERSHIP",
    "TECHNICAL EXPERTISE",
    "PROJECT DELIVERY",
    "POST-SALES SUPPORT"
]

service_label_map = {
    "PRESALES AND PARTNERSHIP": "Presales & Partnership",
    "TECHNICAL EXPERTISE": "Technical Expertise",
    "PROJECT DELIVERY": "Project Delivery",
    "POST-SALES SUPPORT": "Post-Sales Support"
}

segment_order = ["Overall"] + [
    client_type
    for client_type in ["Govt", "NPO", "Private"]
    if client_type in df["TYPE"].unique()
]


# -------------------------------------------------
# 2. Create Overall and client-segment datasets
# -------------------------------------------------

client_groups = {
    "Overall": df
}

for client_type in segment_order[1:]:
    client_groups[client_type] = df[
        df["TYPE"] == client_type
    ]


# -------------------------------------------------
# 3. Calculate Spearman correlations
# -------------------------------------------------

correlation_tables = {}

for segment, group_data in client_groups.items():
    correlations = (
        group_data[service_columns + ["NPS RATING"]]
        .corr(method="spearman")["NPS RATING"]
        .drop("NPS RATING")
        .rename("Correlation")
        .reset_index()
        .rename(columns={"index": "Service Dimension"})
    )

    correlations["Service Dimension"] = (
        correlations["Service Dimension"]
        .replace(service_label_map)
    )

    correlations["Records"] = len(group_data)

    correlations["Unique Clients"] = (
        group_data["CLIENT ID"].nunique()
    )

    # Weakest at the bottom and strongest at the top
    correlations = correlations.sort_values(
        "Correlation",
        ascending=True
    )

    correlation_tables[segment] = correlations


# -------------------------------------------------
# 4. Create Graph Objects figure
# -------------------------------------------------

fig3 = go.Figure()

for index, segment in enumerate(segment_order):
    table = correlation_tables[segment]

    bar_colours = [
        "#2A9D8F" if value >= 0 else "#E76F51"
        for value in table["Correlation"]
    ]

    hover_data = [
        [records, clients]
        for records, clients in zip(
            table["Records"],
            table["Unique Clients"]
        )
    ]

    fig3.add_trace(
        go.Bar(
            x=table["Correlation"],
            y=table["Service Dimension"],

            orientation="h",

            name=segment,
            visible=(index == 0),

            marker=dict(
                color=bar_colours,
                line=dict(
                    color="white",
                    width=1
                )
            ),

            text=[
                f"{value:+.2f}"
                for value in table["Correlation"]
            ],

            textposition="outside",
            cliponaxis=False,

            customdata=hover_data,

            hovertemplate=(
                "<b>%{y}</b><br>"
                f"Client Segment: {segment}<br>"
                "Spearman Correlation: %{x:.3f}<br>"
                "Records Used: %{customdata[0]}<br>"
                "Unique Clients: %{customdata[1]}"
                "<extra></extra>"
            )
        )
    )


# -------------------------------------------------
# 5. Create client-segment dropdown
# -------------------------------------------------

segment_buttons = []

for index, segment in enumerate(segment_order):
    visibility = [False] * len(segment_order)
    visibility[index] = True

    table = correlation_tables[segment]

    segment_buttons.append(
        dict(
            label=segment,
            method="update",
            args=[
                {
                    "visible": visibility
                },
                {
                    "title.text": (
                        "Service Dimensions Associated with NPS"
                        f" — {segment}"
                        "<br><sup>"
                        "Spearman correlation; association does not "
                        "necessarily imply causation"
                        "</sup>"
                    ),
                    "yaxis.categoryorder": "array",
                    "yaxis.categoryarray": (
                        table["Service Dimension"].tolist()
                    )
                }
            ]
        )
    )


# -------------------------------------------------
# 6. Add correlation-strength background regions
# -------------------------------------------------

# Strong negative
fig3.add_vrect(
    x0=-1,
    x1=-0.5,
    fillcolor="#E76F51",
    opacity=0.10,
    line_width=0,
    layer="below"
)

# Moderate negative
fig3.add_vrect(
    x0=-0.5,
    x1=-0.3,
    fillcolor="#F4A261",
    opacity=0.10,
    line_width=0,
    layer="below"
)

# Weak relationship
fig3.add_vrect(
    x0=-0.3,
    x1=0.3,
    fillcolor="#BDBDBD",
    opacity=0.08,
    line_width=0,
    layer="below"
)

# Moderate positive
fig3.add_vrect(
    x0=0.3,
    x1=0.5,
    fillcolor="#F6D365",
    opacity=0.10,
    line_width=0,
    layer="below"
)

# Strong positive
fig3.add_vrect(
    x0=0.5,
    x1=1,
    fillcolor="#2A9D8F",
    opacity=0.10,
    line_width=0,
    layer="below"
)

fig3.add_vline(
    x=0,
    line_width=2,
    line_color="#555555"
)


# -------------------------------------------------
# 7. Format the chart
# -------------------------------------------------

initial_table = correlation_tables["Overall"]

fig3.update_layout(
    title=dict(
        text=(
            "Service Dimensions Associated with NPS — Overall"
            "<br><sup>"
            "Spearman correlation; association does not "
            "necessarily imply causation"
            "</sup>"
        ),
        x=0.5,
        xanchor="center"
    ),

    xaxis=dict(
        title="Spearman Correlation with NPS",
        range=[-1.05, 1.05],
        dtick=0.2,
        zeroline=False
    ),

    yaxis=dict(
        title="Service Dimension",
        categoryorder="array",
        categoryarray=(
            initial_table["Service Dimension"].tolist()
        )
    ),

    updatemenus=[
        dict(
            buttons=segment_buttons,
            direction="down",
            showactive=True,
            active=0,
            x=1,
            xanchor="right",
            y=1.18,
            yanchor="top"
        )
    ],

    annotations=[
        dict(
            text="Select client segment:",
            x=0.72,
            y=1.16,
            xref="paper",
            yref="paper",
            showarrow=False
        )
    ],

    template="plotly_white",
    showlegend=False,
    hovermode="closest",

    margin=dict(
        l=190,
        r=90,
        t=140,
        b=80
    ),

    height=520
)


# -------------------------------------------------
# 8. Display the graph
# -------------------------------------------------

fig3.show(
    config={
        "displayModeBar": True,
        "displaylogo": False,
        "scrollZoom": True,
        "responsive": True,

        "toImageButtonOptions": {
            "format": "png",
            "filename": "service_nps_correlation",
            "scale": 2
        }
    }
)

### Key findings from Visualisation 3

- Overall, **Technical Expertise** has the strongest association with NPS Rating at 0.544, closely followed by **Project Delivery** at 0.538.
- For Govt clients, **Project Delivery** is strongest at 0.559.
- For NPO clients, **Project Delivery** is strongest at 0.670, followed by **Presales & Partnership** at 0.619.
- For Private clients, **Technical Expertise** is strongest at 0.571.
- All displayed relationships are positive: stronger service ratings tend to occur alongside stronger NPS ratings.

**Business implication:** Project Delivery and Technical Expertise are the most consistent loyalty priorities overall, but the best improvement focus differs by client type. Segment-specific actions are therefore more useful than a single company-wide response.

**Caution:** Correlation shows association, not causation. These results do not prove that changing one service rating will directly cause NPS to change, and repeated observations from clients across years should be interpreted descriptively.


## Interactive satisfaction dashboard

The dashboard combines the three visualisations into one decision view:

1. the grouped bar chart compares **service performance**;
2. the heatmap compares **loyalty levels over time**; and
3. the correlation chart highlights potential **loyalty drivers**.

The KPI cards provide an overall snapshot, while the chart controls allow users to explore periods and client types. Run the cell below to launch the dashboard locally.

In [8]:
from copy import deepcopy
from dash import Dash, dcc, html


# -------------------------------------------------
# 1. Create dashboard copies of the graphs
# -------------------------------------------------

service_fig = deepcopy(fig)
nps_fig = deepcopy(fig2)
driver_fig = deepcopy(fig3)

service_fig.update_layout(
    height=600,
    autosize=True
)

nps_fig.update_layout(
    height=500,
    autosize=True
)

driver_fig.update_layout(
    height=500,
    autosize=True
)


# -------------------------------------------------
# 2. Calculate dashboard summary metrics
# -------------------------------------------------

overall_satisfaction = (
    df[service_columns]
    .mean(axis=1)
    .mean()
)

overall_nps = df["NPS RATING"].mean()

service_means = df[service_columns].mean()
strongest_service_column = service_means.idxmax()

strongest_service_name = service_label_map.get(
    strongest_service_column,
    strongest_service_column.title()
)

strongest_service_rating = service_means.max()


# -------------------------------------------------
# 3. Reusable dashboard styles
# -------------------------------------------------

page_style = {
    "backgroundColor": "#F4F6FA",
    "minHeight": "100vh",
    "padding": "28px",
    "fontFamily": "Segoe UI, Arial, sans-serif",
    "color": "#172B4D"
}

content_style = {
    "maxWidth": "1500px",
    "margin": "0 auto"
}

metric_grid_style = {
    "display": "grid",
    "gridTemplateColumns": (
        "repeat(auto-fit, minmax(220px, 1fr))"
    ),
    "gap": "16px",
    "marginBottom": "22px"
}

metric_card_style = {
    "backgroundColor": "white",
    "padding": "20px",
    "borderRadius": "12px",
    "boxShadow": "0 2px 8px rgba(0, 0, 0, 0.08)"
}

chart_card_style = {
    "backgroundColor": "white",
    "padding": "12px",
    "borderRadius": "12px",
    "boxShadow": "0 2px 8px rgba(0, 0, 0, 0.08)"
}

bottom_grid_style = {
    "display": "grid",
    "gridTemplateColumns": (
        "repeat(auto-fit, minmax(480px, 1fr))"
    ),
    "gap": "20px",
    "marginTop": "20px"
}

graph_config = {
    "displaylogo": False,
    "responsive": True,
    "scrollZoom": True,
    "toImageButtonOptions": {
        "format": "png",
        "filename": "client_satisfaction_dashboard",
        "scale": 2
    }
}


# -------------------------------------------------
# 4. Build Dash application
# -------------------------------------------------

app = Dash(__name__)

app.title = "Client Satisfaction Dashboard"

app.layout = html.Div(
    style=page_style,
    children=[
        html.Div(
            style=content_style,
            children=[
                # Dashboard heading
                html.Div(
                    children=[
                        html.H1(
                            "Client Satisfaction and Loyalty Dashboard",
                            style={
                                "marginBottom": "6px",
                                "fontWeight": "600"
                            }
                        ),

                        html.P(
                            (
                                "Explore service satisfaction, "
                                "customer advocacy and the service "
                                "dimensions associated with NPS."
                            ),
                            style={
                                "color": "#5E6C84",
                                "marginTop": "0",
                                "marginBottom": "24px"
                            }
                        )
                    ]
                ),

                # Summary metrics
                html.Div(
                    style=metric_grid_style,
                    children=[
                        html.Div(
                            style=metric_card_style,
                            children=[
                                html.P(
                                    "Overall Service Rating",
                                    style={
                                        "color": "#5E6C84",
                                        "margin": "0"
                                    }
                                ),
                                html.H2(
                                    f"{overall_satisfaction:.2f} / 5",
                                    style={
                                        "marginBottom": "0"
                                    }
                                )
                            ]
                        ),

                        html.Div(
                            style=metric_card_style,
                            children=[
                                html.P(
                                    "Average NPS Rating",
                                    style={
                                        "color": "#5E6C84",
                                        "margin": "0"
                                    }
                                ),
                                html.H2(
                                    f"{overall_nps:.2f} / 10",
                                    style={
                                        "marginBottom": "0"
                                    }
                                )
                            ]
                        ),

                        html.Div(
                            style=metric_card_style,
                            children=[
                                html.P(
                                    "Strongest Service Dimension",
                                    style={
                                        "color": "#5E6C84",
                                        "margin": "0"
                                    }
                                ),
                                html.H3(
                                    strongest_service_name,
                                    style={
                                        "marginBottom": "4px"
                                    }
                                ),
                                html.P(
                                    (
                                        f"Average rating: "
                                        f"{strongest_service_rating:.2f} / 5"
                                    ),
                                    style={
                                        "color": "#5E6C84",
                                        "margin": "0"
                                    }
                                )
                            ]
                        )
                    ]
                ),

                # Main satisfaction overview
                html.Div(
                    style=chart_card_style,
                    children=[
                        dcc.Graph(
                            id="service-rating-chart",
                            figure=service_fig,
                            config=graph_config,
                            responsive=True
                        )
                    ]
                ),

                # NPS and driver analysis
                html.Div(
                    style=bottom_grid_style,
                    children=[
                        html.Div(
                            style=chart_card_style,
                            children=[
                                dcc.Graph(
                                    id="nps-heatmap",
                                    figure=nps_fig,
                                    config=graph_config,
                                    responsive=True
                                )
                            ]
                        ),

                        html.Div(
                            style=chart_card_style,
                            children=[
                                dcc.Graph(
                                    id="nps-driver-chart",
                                    figure=driver_fig,
                                    config=graph_config,
                                    responsive=True
                                )
                            ]
                        )
                    ]
                ),

                html.P(
                    (
                        "Correlations indicate association and do not "
                        "necessarily establish causation."
                    ),
                    style={
                        "textAlign": "center",
                        "color": "#7A869A",
                        "fontSize": "12px",
                        "marginTop": "18px"
                    }
                )
            ]
        )
    ]
)


# -------------------------------------------------
# 5. Run dashboard
# -------------------------------------------------

app.run(debug=True)

## Conclusion and recommendations

The results do not identify one client type as best on every satisfaction measure. Govt clients lead several service averages, while NPO clients have the highest overall NPS Rating by a very small margin. Private clients have the lowest overall NPS Rating and their weakest overall service dimension is Post-Sales Support.

Recommended actions:

1. **Maintain Project Delivery quality**, which is the strongest service dimension across every client type.
2. **Strengthen Private Post-Sales Support** to address a service weakness within the segment with the lowest overall NPS Rating.
3. **Improve NPO Presales & Partnership**, while maintaining NPO's strong Project Delivery and Post-Sales results.
4. Use segment-specific loyalty priorities: Project Delivery for Govt and NPO clients, and Technical Expertise for Private clients.
5. Combine these satisfaction findings with the profitability analysis. Client segments should be prioritised only when they provide both **strong financial value** and **strong customer outcomes**.

This notebook therefore answers the satisfaction component of the main problem and provides the evidence needed for the team's final profit–satisfaction prioritisation.
